# Step 05 — Qwen2.5-3B QLoRA on a Colab T4 with Unsloth

Use a **fresh T4 GPU runtime**. The default model is `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`, loaded in 4-bit with FP16 computation. BF16 and full-precision loading are disabled. Nothing in this notebook runs on your local machine unless you explicitly execute it there.

Defaults: 1,024-token context, batch size 1, accumulation 16, paged 8-bit AdamW, rank/alpha 16, zero dropout, Unsloth gradient checkpointing, and no periodic validation. Complete training/validation examples that exceed the context budget are excluded and reported; target JSON is never cut. If too few examples fit, the notebook stops before training and explains how to adjust the context budget.

The compact prompt omits the verbose JSON schema. The target JSON still follows the project schema, and inference uses the same prompt. Old Step 04 scores used a different prompt and are not a direct baseline. Optional before/after evaluation below uses matched records/settings.

Checkpoints are committed to Hugging Face by SHA; runtime disk is temporary staging. No Google Drive mount is used. Setup, recovery controls, and troubleshooting are included below. Use this notebook and its Python module from the same updated checkout. GPU fit and model quality still require observation in Colab.

The installation cell pins the Colab training stack separately from the root requirements and prints installed versions before loading weights. These pins were checked against upstream declarations; installation, CUDA compatibility, and training still need verification in Colab. If cloning from GitHub, make sure the notebook and module changes are available in that checkout.


In [ ]:
# Select Runtime > Change runtime type > T4 GPU, then run cells from top to bottom.
# After an OOM or an old Transformers/TRL import, disconnect/delete the runtime first.

In [ ]:
# Sync the Colab checkout; preserve an existing local checkout.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'
CLONE_DIR = Path('/content/resume-parser-model')
MARKER = 'src/steps/05_fine_tuning/fine_tuning.py'
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if (CLONE_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(CLONE_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)], check=True)
    PROJECT_ROOT = CLONE_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / MARKER).is_file():
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            raise FileNotFoundError('Open this notebook from inside the project.')
        PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
# Install Colab dependencies here, preserving the existing CUDA Torch wheel.
import subprocess
import sys
from collections import deque
from importlib.metadata import version, PackageNotFoundError

if sys.platform != 'linux':
    raise RuntimeError('Run this GPU dependency installation in Colab, not on the local computer.')
try:
    torch_version = version('torch').split('+')[0]
except PackageNotFoundError:
    raise RuntimeError('Use a Colab GPU runtime with CUDA PyTorch preinstalled.')

COLAB_PACKAGES = [
    'unsloth[huggingfacenotorch]==2026.10.3',
    'unsloth_zoo>=2026.10.6,<2026.11.0',
    'transformers==5.17.0',
    'trl==1.15.0',
    'datasets==4.3.0',
    'accelerate==1.15.0',
    'peft==0.21.2',
    'bitsandbytes==0.50.2',
    'wandb==0.30.0',
    'python-dotenv>=1.0.0',
    'pydantic>=2.7.0',
    'tqdm>=4.66.0',
    'matplotlib>=3.8.0',
    'typesafe-sdk>=0.7.0',
]
# Stream pip output through the notebook and include its final lines in any failure.
# CalledProcessError alone only reports an exit status; it does not explain the cause.
print('Python:', sys.version.split()[0], '| executable:', sys.executable, flush=True)
print('Preserving installed Torch:', torch_version, flush=True)
pip_command = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--progress-bar', 'off',
    f'torch=={torch_version}', *COLAB_PACKAGES,
]
pip_tail = deque(maxlen=80)
with subprocess.Popen(
    pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        pip_tail.append(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f'Colab dependency installation failed (pip exit code {return_code}). '
        'Do not run the model/trainer cells. Share the ERROR lines below so the '
        'failing package or dependency conflict can be identified.\n\n'
        + ''.join(pip_tail)
    )
print('Dependency installation completed.', flush=True)
# If installation fails, use a fresh T4 runtime and retry this cell once.
# Do not install the root requirements.txt afterward; it has different training pins.
# If it still fails, keep the error and follow the official Unsloth Colab install guide:
# https://unsloth.ai/docs/get-started/install/google-colab


In [ ]:
# Import Unsloth before Transformers/PEFT/TRL, then inspect the actual installed APIs.
import os
import sys
import json
import inspect
from importlib import import_module, reload
from importlib.metadata import version

if 'unsloth' not in sys.modules and any(name in sys.modules for name in ('transformers', 'peft', 'trl')):
    raise RuntimeError('An older training stack is already imported. Start a fresh runtime and run from the top.')
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=False)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
fine_tuning = reload(import_module('src.steps.05_fine_tuning.fine_tuning'))
required_helpers = ('apply_lora', 'render_training_prompt', 'build_hub_checkpoint_callback', 'download_checkpoint_from_hub')
missing = [name for name in required_helpers if not callable(getattr(fine_tuning, name, None))]
if missing:
    raise RuntimeError(f'Update the checkout: Step 05 is missing {missing}.')
report_results = import_module('src.steps.04_evaluation').report_results
runtime_versions = {}
for package in ('torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'accelerate', 'datasets', 'bitsandbytes', 'wandb', 'tokenizers', 'pydantic', 'huggingface_hub', 'triton'):
    installed = version(package)
    runtime_versions[package] = installed
    loaded = getattr(sys.modules.get(package), '__version__', installed)
    if str(loaded) != installed:
        raise RuntimeError(f'{package} changed after import. Start a fresh runtime.')
    print(f'{package}: {installed}')
print('Trainer signature:', inspect.signature(SFTTrainer.__init__))
fields = getattr(SFTConfig, '__dataclass_fields__', {})
if not {'max_length', 'completion_only_loss'} <= set(fields):
    raise RuntimeError('Unexpected TRL API. Rerun the notebook installation cell in a fresh runtime.')
print('Fine-tuning implementation:', fine_tuning.__file__)

### Secrets -> environment variables

Pulls the project keys from **Google Colab's Secrets manager** (the key icon in the left sidebar) and exposes them as environment variables for every later cell. Colab values take precedence over `.env`. Outside Colab this is a no-op, so the `.env` file is used instead.

In [ ]:
# Keys this step reads from Colab Secrets (or .env). Repo ids live in the code,
# not here. Extras are skipped harmlessly.
SECRET_KEYS = [
    'HF_TOKEN',             # checkpoint uploads + optional adapter publication
    'HF_DATASET_REVISION',  # optional: dataset revision (defaults to main)
    'WANDB_API_KEY',        # Weights & Biases login
    'WANDB_ENTITY',         # optional: W&B entity (has a default)
    'WANDB_PROJECT',        # optional: W&B project (has a default)
    'TYPESAFE_API_KEY',     # step-04 semantic judge during evaluation
]


def load_colab_secrets(keys=SECRET_KEYS):
    try:
        from google.colab import userdata  # available only inside Colab
    except ImportError:
        print('Not running in Colab; using .env / existing environment instead.')
        return {}
    loaded = {}
    for key in keys:
        try:
            value = userdata.get(key)
        except Exception:
            continue  # not defined or notebook access not granted
        if value:
            os.environ[key] = value  # Colab Secrets take precedence over .env
            loaded[key] = value
    return loaded


colab_secrets = load_colab_secrets()
print('Secrets loaded from Colab:', ', '.join(sorted(colab_secrets)) or 'none')

In [ ]:
# Reporting and publishing can be disabled without requiring credentials.
USE_WANDB = False  # Enable only when you want experiment tracking.
HUB_CHECKPOINTS = True  # Commit every complete checkpoint to Hugging Face.
PUSH_TO_HUB = False
RUN_BASELINE = False  # Set True for a directly comparable before/after evaluation.

required_keys = []
if USE_WANDB:
    required_keys.append('WANDB_API_KEY')
if HUB_CHECKPOINTS or PUSH_TO_HUB:
    required_keys.append('HF_TOKEN')
for key in SECRET_KEYS:
    print(f'{key}:', 'set' if os.getenv(key) else 'unset')
missing = [key for key in required_keys if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing required keys: {', '.join(missing)}. Add them to Colab Secrets or .env.")
if not os.getenv('TYPESAFE_API_KEY'):
    print('Semantic evaluation is unavailable. Inspect programmatic metrics and semantic-error statuses.')

In [ ]:
# Quick validation uses a small sample and cannot establish global extraction quality.
MODEL_ID = fine_tuning.DEFAULT_MODEL_ID
LOAD_IN_4BIT = True
EVAL_SAMPLE = 3
MAX_NEW_TOKENS = 1024  # Also capped to the selected context window minus prompt length.
WANDB_ENTITY = os.getenv('WANDB_ENTITY')
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')
print('Base model:', MODEL_ID)
print('HF model repo:', fine_tuning.model_repo_id())
print('HF dataset repo:', fine_tuning.dataset_repo_id())

### Hyperparameters and recovery

Begin at 1,024 tokens. Length reports show how many examples fit each candidate budget. Use 512 only when complete examples fit; larger budgets admit longer resumes but use more VRAM. The compact prompt has no inline schema, so use the same prompt for deployment.

Keep `RUN_ID=None` for a new experiment. After a runtime reset, set the original `RUN_ID` and `RESUME_HUB_REVISION` to a confirmed full checkpoint SHA. Resume requires the same model, backend, prompt, hyperparameters, precision, input hashes, and package versions. Old Transformers/schema-prompt checkpoints require a new experiment for this profile.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

HYPERPARAMETERS = {
    'learning_rate': 2e-4,
    'epochs': 2,
    'per_device_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'max_seq_length': 1024,
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0,
    'target_modules': list(fine_tuning.DEFAULT_TARGET_MODULES),
    'optim': 'paged_adamw_8bit',
    'evaluate_during_training': False,
    'warmup_ratio': 0.03,
    'lr_scheduler_type': 'cosine',
    'save_steps': 50,
    'eval_steps': 50,
    'max_train_samples': 2000,  # Stratified selection after cleaning and length checks.
    'seed': 42,
}
OVERLENGTH_POLICY = 'drop'  # Exclusions are reported; no target JSON is truncated.
PROMPT_STYLE = 'compact'  # 'schema' is the old, much longer prompt.
MIN_TRAIN_EXAMPLES = 64
MIN_VALIDATION_EXAMPLES = 8
RUN_TAG = 'unsloth-t4-1k-r16'
RUN_ID = None  # To resume, paste the previous run ID here.
RESUME_HUB_REVISION = None  # Full commit SHA (recommended), or original experiment branch.
RESUME_FROM_CHECKPOINT = None  # Only for local recovery: True = latest; or a checkpoint path.

if RESUME_HUB_REVISION and RESUME_FROM_CHECKPOINT:
    raise ValueError('Choose either Hub recovery or local recovery.')
RECOVERING = bool(RESUME_HUB_REVISION or RESUME_FROM_CHECKPOINT)
if RECOVERING and not RUN_ID:
    raise ValueError('Set the original RUN_ID before resuming.')
EXPERIMENT_ID = RUN_ID or f"{RUN_TAG}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex[:8]}"
RUN_NAME = f'qwen2.5-3b-resume-parser-qlora-{EXPERIMENT_ID}'
# Runtime disk stages checkpoints; Hugging Face holds durable versions.
output_root = fine_tuning.MODELS_DIR
OUTPUT_DIR = output_root / RUN_NAME
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not RECOVERING:
    raise ValueError('Output directory already contains a run. Choose a new RUN_ID or resume explicitly.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_HUB_REVISION:
    RESUME_FROM_CHECKPOINT = str(fine_tuning.download_checkpoint_from_hub(
        OUTPUT_DIR, repo_id=fine_tuning.model_repo_id(), revision=RESUME_HUB_REVISION,
    ))
print('Run ID (save for recovery):', EXPERIMENT_ID)
print('Output dir:', OUTPUT_DIR)
print(json.dumps(HYPERPARAMETERS, indent=2))
print('Hugging Face checkpoint uploads:', 'enabled' if HUB_CHECKPOINTS else 'disabled (local only)')

## 1. Download, validate, and prepare the splits

The test split is downloaded only to reserve its resume identities/text. It supplies no training targets. Training excludes all held-out overlap; duplicate texts are deduplicated, and groups with conflicting targets are excluded with counts. Original dataset files are preserved.

In [ ]:
splits = fine_tuning.download_curated_splits(splits=('train', 'validation', 'test'))
raw_train_records = fine_tuning.load_records(splits['train'])
raw_validation_records = fine_tuning.load_records(splits['validation'])
test_records = fine_tuning.load_records(splits['test'])
train_records, validation_records, preparation_report = fine_tuning.prepare_training_records(
    raw_train_records, raw_validation_records, test_records=test_records,
)
print(json.dumps(preparation_report, indent=2))

In [ ]:
# Inspect the actual GPU and release only unused allocations before loading weights.
import gc
subprocess.run(['nvidia-smi'], check=True)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a T4 runtime.')
for name in ('trainer', 'model', 'tokenizer', 'checkpoint_callback', 'callback'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print('GPU:', torch.cuda.get_device_name(0))
print('Total VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Training precision: fp16=True, bf16=False')

## 2. Load a fresh 4-bit model

Run the GPU cleanup cell first. `empty_cache()` only releases unused allocations; it cannot free a model held by old variables or an exception traceback. Use a fresh runtime after OOM. The loader uses FP16 explicitly and never loads full-precision model weights.

In [ ]:
# Load the prequantized model with the same context limit used by dataset filtering.
model, tokenizer = fine_tuning.load_model_and_tokenizer(
    MODEL_ID, load_in_4bit=LOAD_IN_4BIT, max_seq_length=HYPERPARAMETERS['max_seq_length'],
)
print('Loaded:', MODEL_ID, '| compute dtype: FP16 | 4-bit: True')

## 3. Apply LoRA and prepare complete examples

Unsloth attaches adapters with `use_gradient_checkpointing="unsloth"`. Dataset preparation retains conversational prompt/completion columns so TRL masks the prompt and trains only on the JSON response. Length checks include the Qwen chat template and the whole target JSON.

In [ ]:
# Attach fresh Unsloth adapters exactly once.
import hashlib
from functools import partial
from collections import Counter
prompt_fn = partial(fine_tuning.render_training_prompt, style=PROMPT_STYLE)
model = fine_tuning.apply_lora(
    model, r=HYPERPARAMETERS['lora_r'], alpha=HYPERPARAMETERS['lora_alpha'],
    dropout=HYPERPARAMETERS['lora_dropout'], target_modules=HYPERPARAMETERS['target_modules'],
    seed=HYPERPARAMETERS['seed'],
)
model.print_trainable_parameters()



In [ ]:
# Measure complete sequences and build completion-only datasets.
for name, records in [('train', train_records), ('validation', validation_records)]:
    lengths = fine_tuning.token_length_report(
        records, tokenizer, sample=None, max_length=HYPERPARAMETERS['max_seq_length'], prompt_fn=prompt_fn,
    )
    print(f'{name} complete-example length report:', json.dumps(lengths, indent=2))
fit_train_records, train_lengths = fine_tuning.filter_records_by_length(
    train_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'],
    policy=OVERLENGTH_POLICY, prompt_fn=prompt_fn, min_samples=MIN_TRAIN_EXAMPLES,
)
fit_validation_records, validation_lengths = fine_tuning.filter_records_by_length(
    validation_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'],
    policy=OVERLENGTH_POLICY, prompt_fn=prompt_fn, min_samples=MIN_VALIDATION_EXAMPLES,
)
print('Training exclusions:', train_lengths)
print('Validation exclusions:', validation_lengths)
train_dataset, eval_dataset = fine_tuning.build_datasets(
    fit_train_records, fit_validation_records, tokenizer=tokenizer,
    max_train=HYPERPARAMETERS['max_train_samples'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    seed=HYPERPARAMETERS['seed'], test_records=test_records, prompt_fn=prompt_fn,
)

# Hugging Face revision for this round, keyed to how many resumes were trained.
NUM_TRAIN_RESUMES = len(train_dataset)
REVISION = fine_tuning.checkpoint_revision(NUM_TRAIN_RESUMES, run_tag=EXPERIMENT_ID)
print(train_dataset)
print(eval_dataset)
print('Trained resumes:', NUM_TRAIN_RESUMES, '| HF revision:', REVISION)
selected_records = fine_tuning.select_training_records(
    fit_train_records, HYPERPARAMETERS['max_train_samples'], seed=HYPERPARAMETERS['seed'],
)
from collections import Counter
print('Selected category counts:', dict(Counter(r.get('category') for r in selected_records)))


## 4. Train and retain resumable checkpoints

For another experiment, rerun the configuration, fresh model-loading, dataset, and training cells. To resume, keep the original run ID and manifest settings. Checkpoints are saved every `save_steps` and committed synchronously to Hugging Face. Interruptions can lose progress since the latest confirmed commit SHA. Local retention (`save_total_limit`) does not delete older Hub commit history.

Optional baseline evaluation runs on exactly the same prepared validation records, sample seed, and output-token budget as post-training evaluation. Old charts under `evals/` may use different samples/splits and are not a direct before/after comparison.

In [ ]:
# Configure FP16 training and record the exact inputs needed for checkpoint recovery.
training_args = fine_tuning.build_training_arguments(
    OUTPUT_DIR,
    epochs=HYPERPARAMETERS['epochs'],
    learning_rate=HYPERPARAMETERS['learning_rate'],
    per_device_batch_size=HYPERPARAMETERS['per_device_batch_size'],
    per_device_eval_batch_size=HYPERPARAMETERS['per_device_eval_batch_size'],
    gradient_accumulation_steps=HYPERPARAMETERS['gradient_accumulation_steps'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    save_steps=HYPERPARAMETERS['save_steps'],
    eval_steps=HYPERPARAMETERS['eval_steps'],
    warmup_ratio=HYPERPARAMETERS['warmup_ratio'],
    lr_scheduler_type=HYPERPARAMETERS['lr_scheduler_type'],
    run_name=RUN_NAME,
    use_wandb=USE_WANDB, bf16=False,
    optim=HYPERPARAMETERS['optim'],
    evaluate_during_training=HYPERPARAMETERS['evaluate_during_training'],
    seed=HYPERPARAMETERS['seed'],
)
manifest = {
    'backend': 'unsloth', 'prompt_style': PROMPT_STYLE,
    'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'hyperparameters': HYPERPARAMETERS,
    'runtime_versions': runtime_versions,
    'training_precision': {'bf16': training_args.bf16, 'fp16': training_args.fp16},
    'prompt_sha256': hashlib.sha256(prompt_fn('STATIC_PROMPT_MARKER').encode('utf-8')).hexdigest(),
    'schema_sha256': hashlib.sha256((PROJECT_ROOT / 'src/schema/resume_output.py').read_bytes()).hexdigest(),
    'dataset_revision': os.getenv('HF_DATASET_REVISION', 'main'),
    'split_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in splits.items()},
    'train_resume_ids': [r['resume_id'] for r in selected_records],
    'preparation': preparation_report, 'train_lengths': train_lengths, 'validation_lengths': validation_lengths,
}
manifest_path = OUTPUT_DIR / 'run_manifest.json'
if RESUME_FROM_CHECKPOINT:
    previous = json.loads(manifest_path.read_text())
    for key in ('backend', 'prompt_style', 'run_id', 'model_id', 'hyperparameters', 'split_sha256', 'train_resume_ids', 'runtime_versions', 'training_precision', 'prompt_sha256', 'schema_sha256'):
        if previous.get(key) != manifest[key]:
            raise ValueError(f'Resume configuration changed: {key}. Use a new RUN_ID for the Unsloth/compact-prompt profile.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
checkpoint_callback = None
if HUB_CHECKPOINTS:
    checkpoint_callback = fine_tuning.build_hub_checkpoint_callback(
        repo_id=fine_tuning.model_repo_id(), revision=REVISION,
    )
    print('Checkpoint branch:', REVISION)
    print('Recover by full SHA at:', f'https://huggingface.co/{fine_tuning.model_repo_id()}/commits/{REVISION}')

In [ ]:
if RUN_BASELINE and not RESUME_FROM_CHECKPOINT:
    baseline_report = fine_tuning.evaluate_records(
        model, tokenizer, fit_validation_records, model_name=f'baseline:{MODEL_ID}',
        limit=min(EVAL_SAMPLE, len(fit_validation_records)), max_new_tokens=MAX_NEW_TOKENS,
        max_seq_length=HYPERPARAMETERS['max_seq_length'], prompt_fn=prompt_fn,
        seed=HYPERPARAMETERS['seed'],
        output_dir=OUTPUT_DIR / 'evaluation-before',
    )
    print('Baseline:', json.dumps(baseline_report.metrics, indent=2))

if USE_WANDB:
    import wandb
    if wandb.run is not None:
        wandb.finish()
    fine_tuning.configure_wandb_env(entity=WANDB_ENTITY, project=WANDB_PROJECT, run_name=RUN_NAME)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
training_succeeded = False
try:
    trainer = fine_tuning.train(
        model, tokenizer, train_dataset, eval_dataset,
        training_args=training_args,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
        callbacks=[checkpoint_callback] if checkpoint_callback is not None else None,
    )
    training_succeeded = True
finally:
    if USE_WANDB:
        wandb.finish(exit_code=0 if training_succeeded else 1)
if checkpoint_callback is not None:
    print('Confirmed checkpoint commits:')
    for uploaded in checkpoint_callback.uploads:
        print(uploaded['global_step'], uploaded['commit_sha'], uploaded['commit_url'])

In [ ]:
# Peak allocator memory includes model weights and training tensors; nvidia-smi also includes CUDA overhead.
print(f'Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB')
print(f'Peak reserved GPU memory: {torch.cuda.max_memory_reserved() / 2**30:.2f} GiB')

## 5. Quick inference and small-sample evaluation

The quick inference check uses synthetic resume text. Full evaluation uses the same compact prompt as training, a small sample, and a generation budget capped to available context. It is not directly comparable to old schema-prompt baseline charts. Full semantic scoring still needs `TYPESAFE_API_KEY`.

In [ ]:
# Use synthetic text for a quick generation check without printing private resumes.
quick_item = fine_tuning.EvaluationItem(
    resume_id='synthetic-smoke-check',
    resume_text='Alex Example is a Python developer. Worked at Example Company from 2021 to 2024. Skills: Python, SQL.',
    target_json={},
)
callback = fine_tuning.make_generation_callback(
    trainer.model, tokenizer, model_name='unsloth-qlora', prompt_fn=prompt_fn,
    max_seq_length=HYPERPARAMETERS['max_seq_length'], max_new_tokens=MAX_NEW_TOKENS,
)
quick_output = callback(quick_item)
print(quick_output.raw_output)
try:
    quick_json = json.loads(quick_output.raw_output)
    from src.schema.resume_output import ResumeOutput
    ResumeOutput.model_validate(quick_json)
    print('Synthetic output is valid JSON matching ResumeOutput.')
except (ValueError, TypeError):
    print('Synthetic output is incomplete or invalid. Inspect the output and training length coverage.')

In [ ]:
adapter_dir = fine_tuning.save_adapter(trainer, tokenizer, OUTPUT_DIR)
print('Saved adapter to:', adapter_dir)

In [ ]:
report = fine_tuning.evaluate_records(
    trainer.model, tokenizer, fit_validation_records,
    model_name=f'finetuned:{MODEL_ID}', limit=min(EVAL_SAMPLE, len(fit_validation_records)),
    max_new_tokens=MAX_NEW_TOKENS, max_seq_length=HYPERPARAMETERS['max_seq_length'],
    prompt_fn=prompt_fn, seed=HYPERPARAMETERS['seed'],
    output_dir=OUTPUT_DIR / 'evaluation-after',
)
print(json.dumps(report.metrics, indent=2))
summary = report_results({f'finetuned:{MODEL_ID}': report})

## 6. Optional adapter publication

Set `PUSH_TO_HUB=True` in the reporting configuration to upload the adapter and tokenizer. Each experiment has a distinct revision containing its run ID. Checkpoint uploads are controlled separately by `HUB_CHECKPOINTS` and already include optimizer/scheduler/RNG state. The final adapter export is for inference and does not contain that resume state.

In [ ]:
if PUSH_TO_HUB:
    url = fine_tuning.push_adapter_to_hub(
        trainer.model, tokenizer, revision=REVISION,
        commit_message=f'Adapter trained on {NUM_TRAIN_RESUMES} resumes ({EXPERIMENT_ID})',
    )
    print(url)
else:
    print('Adapter saved locally; Hub publication is disabled.')

In [ ]:
# Run after evaluation/export; delete references before emptying CUDA cache.
for name in ('trainer', 'model', 'tokenizer', 'callback', 'checkpoint_callback'):
    globals().pop(name, None)
fine_tuning.free_memory()

## If Colab still runs out of memory

Use a fresh runtime after an OOM, then change one setting at a time and use a new run ID:

1. Reduce `max_seq_length` from 1024 to 512 **only if the complete-example report shows enough usable data**. Raising it to 2048/4096 admits longer resumes but increases memory use.
2. Reduce `lora_r` from 16 to 8, then 4; set `lora_alpha` to the same value for this profile.
3. Reduce `target_modules` to `['q_proj', 'v_proj']`.
4. Switch `MODEL_ID` to `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` and use a distinct run ID/repository revision.

Keep batch size 1, FP16, 4-bit loading, Unsloth checkpointing, and no periodic evaluation. Gradient accumulation changes the effective batch size; lowering it generally does not fix the memory needed by one sequence. `empty_cache()` cannot make live tensors disappear.

A small context budget can discard most long resumes or stop preparation entirely. Do not silently truncate target JSON to force training to proceed. Review retained counts/category coverage before interpreting model scores. GPU fit, installation, and model quality must be verified in Colab; this notebook was edited with static checks only.